# Khám phá dữ liệu WESAD — toàn bộ ngực và cổ tay

Phân tích bộ `all_sensors_raw_stats_v1`: cửa sổ 60 giây, bước 5 giây,
majority ≥90%, 112 thống kê raw (63 ngực + 49 cổ tay), 8 metadata.
Bao gồm ECG/EDA/EMG/Resp/Temp/ACC ngực và BVP/EDA/TEMP/ACC cổ tay.
Đây chưa phải bộ HRV/SCR hoặc đặc trưng sinh lý theo schema nháp.

**Giữ nguyên đường dẫn của notebook đã gửi:** đầu vào tại
`data/features/all_sensors_raw_stats_v1/all_subjects.csv`; bảng/hình tại
`notebooks/reports/tables/eda_step5` và `notebooks/reports/figures/eda_step5`.
Đường dẫn PKL vẫn lấy từ `configs/local.yaml` hoặc biến override đã có.

Chọn kernel `.venv`, Restart Kernel → Run All. Cần numpy, pandas,
matplotlib, IPython; PyYAML nếu đọc local.yaml. Nếu chưa có feature,
chạy từ gốc `wesad-stress-detection`:

```powershell
python scripts/extract_features.py --config configs/features_all_sensors.yaml --local configs/local.yaml --resume
```

Notebook không trích lại feature. CSV phải là bảng đủ cảm biến do extractor
mới sinh ra; đổi tên/thư mục bảng 21 feature không tạo được 112 feature.
Output cũ đã xóa để không mang số liệu của phiên bản trước vào báo cáo.
Các hình phân bố được chia trang để phân tích **toàn bộ** feature.


## 1. Thiết lập đường dẫn và phạm vi


In [ ]:
from pathlib import Path
import hashlib
import json
import warnings
from datetime import datetime, timezone
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

# Chỉ sửa biến này nếu notebook không tìm được gốc dự án.
PROJECT_ROOT_OVERRIDE = None  # Hoặc Path(r"C:/duong/dan/wesad-stress-detection")  
if PROJECT_ROOT_OVERRIDE:
    PROJECT_ROOT = Path(PROJECT_ROOT_OVERRIDE).expanduser().resolve()
else:
    candidates = [Path.cwd(), *Path.cwd().parents]
    candidates += [p / 'wesad-stress-detection' for p in list(candidates)]
    PROJECT_ROOT = next((p for p in candidates if (p / 'src/wesad').is_dir() and (p / 'configs').is_dir()), None)
    if PROJECT_ROOT is None:
        raise FileNotFoundError('Không tìm thấy gốc dự án. Hãy sửa PROJECT_ROOT_OVERRIDE ở trên.')

# Chỉ sửa override nếu feature nằm ngoài thư mục mặc định của dự án.
FEATURE_DIR_OVERRIDE = None  # Ví dụ: r'D:/shared/all_sensors_raw_stats_v1'
FEATURE_DIR = (Path(FEATURE_DIR_OVERRIDE).expanduser().resolve() if FEATURE_DIR_OVERRIDE
               else PROJECT_ROOT / 'data/features/all_sensors_raw_stats_v1')
WINDOW_SECONDS = 60
STEP_SECONDS = 5
MIN_LABEL_FRACTION = 0.90
FEATURE_PATH = FEATURE_DIR / 'all_subjects.csv'
TABLE_DIR = PROJECT_ROOT / 'notebooks/reports/tables/eda_step5'
FIG_DIR = PROJECT_ROOT / 'notebooks/reports/figures/eda_step5'
for folder in (TABLE_DIR, FIG_DIR):
    folder.mkdir(parents=True, exist_ok=True)
EXPECTED_SUBJECTS = [f'S{i}' for i in [2,3,4,5,6,7,8,9,10,11,13,14,15,16,17]]
LABEL_NAMES = {1:'Baseline', 2:'Stress', 3:'Amusement'}
COLORS = ['#4c78a8', '#e45756', '#72b7b2']
plt.rcParams.update({'figure.dpi':110, 'axes.spines.top':False, 'axes.spines.right':False})

def save_table(frame, name, index=False):
    frame.to_csv(TABLE_DIR / name, index=index, encoding='utf-8-sig')

def save_figure(fig, name):
    fig.tight_layout()
    fig.savefig(FIG_DIR / name, dpi=180, bbox_inches='tight')
    plt.show()
    plt.close(fig)


## 2. Nạp dữ liệu và schema toàn bộ cảm biến

Đọc feature_columns từ schema sinh cùng gói CSV; đối chiếu đủ 112 thống kê
raw, các nhóm cảm biến và config combined 60/5, majority ≥90%.
Nếu thiếu schema, suy ra theo các cột chest_/wrist_ và ghi cảnh báo.
Nếu thiếu config.json, chưa xác minh được cấu hình tạo bảng từ metadata.
Không dùng schema nháp 68 feature trong configs/ cho bảng raw-stat này.


In [ ]:
if not FEATURE_PATH.is_file():
    raise FileNotFoundError(f'Chưa có {FEATURE_PATH}. Chạy extractor all-sensors ở đầu notebook; đường dẫn được giữ nguyên.')
import csv
with FEATURE_PATH.open(encoding='utf-8-sig', newline='') as stream:
    header = next(csv.reader(stream), [])
if not header or len(header) != len(set(header)):
    raise ValueError('CSV thiếu header hoặc có tên cột trùng.')
df = pd.read_csv(FEATURE_PATH, dtype={'subject_id':str, 'window_id':str})
if df.empty:
    raise ValueError('Bảng feature rỗng.')
required = ['subject_id','window_id','start_s','end_s','label_original',
            'label_binary','label_fraction','quality_flag']
missing_metadata = sorted(set(required) - set(df.columns))
if missing_metadata:
    raise ValueError(f'Thiếu metadata: {missing_metadata}.')
if df[required].isna().any().any():
    raise ValueError('Metadata bắt buộc bị thiếu.')

SENSOR_SPECS = {
    'chest': {'ECG': (700,1), 'EDA': (700,1), 'EMG': (700,1),
              'Resp': (700,1), 'Temp': (700,1), 'ACC': (700,3)},
    'wrist': {'BVP': (64,1), 'EDA': (4,1), 'TEMP': (4,1), 'ACC': (32,3)},
}
STAT_NAMES = ['mean','std','min','max','rms','iqr','range']
expected_features = []
sensor_feature_groups = {}
for device, sensors in SENSOR_SPECS.items():
    for sensor, (rate, axes) in sensors.items():
        prefix = f'{device}_{sensor.lower()}'
        channels = [prefix] if axes == 1 else [f'{prefix}_{axis}' for axis in ['x','y','z','magnitude']]
        columns = [f'{channel}_{stat}' for channel in channels for stat in STAT_NAMES]
        expected_features.extend(columns)
        sensor_feature_groups[prefix] = columns

schema = None
if (FEATURE_DIR / 'feature_schema.json').is_file():
    schema = json.loads((FEATURE_DIR / 'feature_schema.json').read_text(encoding='utf-8'))
    feature_cols = schema.get('feature_columns')
    if (not isinstance(feature_cols, list) or not feature_cols
            or any(not isinstance(c,str) for c in feature_cols)
            or len(feature_cols) != len(set(feature_cols))):
        raise ValueError('feature_columns trong schema rỗng, trùng hoặc sai định dạng.')
else:
    warnings.warn('Thiếu schema đi kèm: suy ra feature từ tiền tố chest_/wrist_ và đối chiếu bộ raw-stat 112 cột.')
    feature_cols = [c for c in df if c.startswith(('chest_','wrist_'))]
missing_features = sorted(set(expected_features) - set(feature_cols))
unexpected_features = sorted(set(feature_cols) - set(expected_features))
missing_csv = sorted(set(feature_cols) - set(df.columns))
if missing_features or unexpected_features or missing_csv:
    raise ValueError(f'Bảng/schema không phải bộ raw-stat đầy đủ. Thiếu feature: {missing_features}; ngoài bộ 112: {unexpected_features}; thiếu trong CSV: {missing_csv}. Cần trích lại đúng bộ toàn cảm biến.')
feature_groups = {
    'chest': [c for c in feature_cols if c.startswith('chest_')],
    'wrist': [c for c in feature_cols if c.startswith('wrist_')],
    'combined': feature_cols,
}
if schema is not None and 'sensor_groups' in schema:
    for group, cols in feature_groups.items():
        declared = schema['sensor_groups'].get(group, [])
        if len(declared) != len(set(declared)) or set(declared) != set(cols):
            raise ValueError(f'Nhóm {group} trong schema không khớp feature_columns.')
extra_columns = sorted(set(df.columns) - set(required + feature_cols))
if extra_columns:
    warnings.warn(f'Các cột ngoài schema không đưa vào phân tích feature: {extra_columns}')
for col in feature_cols + ['start_s','end_s','label_original','label_binary','label_fraction']:
    df[col] = pd.to_numeric(df[col], errors='raise')
if not df.label_original.isin([1,2,3]).all():
    raise ValueError('Nhãn ba lớp chỉ nhận 1, 2, 3.')
if not np.array_equal(df.label_binary.to_numpy(), (df.label_original == 2).astype(int).to_numpy()):
    raise ValueError('label_binary phải là 2→1, 1/3→0.')
if not np.isfinite(df[['start_s','end_s','label_fraction']].to_numpy()).all():
    raise ValueError('Thời gian hoặc tỷ lệ nhãn không hữu hạn.')
if not df.label_fraction.between(MIN_LABEL_FRACTION - 1e-12, 1 + 1e-12).all():
    raise ValueError('label_fraction ngoài [0.90, 1].')
if (df.start_s < 0).any() or not np.isclose(df.end_s-df.start_s, WINDOW_SECONDS, atol=1e-6, rtol=0).all():
    raise ValueError('Thời gian không hợp lệ hoặc cửa sổ không dài 60 giây.')
if not np.isclose(df.start_s/STEP_SECONDS, np.round(df.start_s/STEP_SECONDS), atol=1e-6, rtol=0).all():
    raise ValueError('Có cửa sổ ngoài lưới 5 giây.')
ordered = df.sort_values(['subject_id','start_s'])
gaps = ordered.groupby('subject_id').start_s.diff().dropna()
if (gaps < STEP_SECONDS - 1e-6).any():
    raise ValueError('Có thời điểm bắt đầu trùng hoặc cách nhau dưới 5 giây trong cùng subject.')

processing = None
if (FEATURE_DIR / 'config.json').is_file():
    snapshot = json.loads((FEATURE_DIR / 'config.json').read_text(encoding='utf-8'))
    processing = snapshot.get('processing', snapshot)
    expected_config = {'device':'combined', 'sampling_rate_hz':700,
        'labels':[1,2,3], 'window_seconds':60, 'step_seconds':5,
        'label_rule':'majority', 'min_label_fraction':0.90,
        'feature_version':'all_sensors_raw_stats_v1',
        'subject_processor':'wesad.multisensor:process_subject'}
    for key, expected in expected_config.items():
        if processing.get(key) != expected:
            raise ValueError(f'config.json: {key}={processing.get(key)!r}, cần {expected!r}.')
    declared = processing.get('sensors')
    if not isinstance(declared, dict) or set(declared) != set(SENSOR_SPECS):
        raise ValueError('Config phải có đủ nhóm chest và wrist.')
    for device, sensors in SENSOR_SPECS.items():
        names = declared.get(device)
        if not isinstance(names,list) or len(names)!=len(sensors) or set(names)!=set(sensors):
            raise ValueError(f'Config chưa chọn đủ cảm biến {device}.')
    print('Đã đối chiếu metadata: combined 60/5, majority ≥90%, toàn bộ cảm biến raw-stat.')
else:
    warnings.warn('Thiếu config.json: CSV phù hợp lưới/tỷ lệ nhãn nhưng chưa xác minh cấu hình trích xuất.')
subject_order = sorted(df.subject_id.unique(), key=lambda s: (int(s[1:]) if s[1:].isdigit() else 999, s))
missing_subjects = sorted(set(EXPECTED_SUBJECTS)-set(subject_order))
extra_subjects = sorted(set(subject_order)-set(EXPECTED_SUBJECTS))
if missing_subjects or extra_subjects:
    warnings.warn(f'Phạm vi khác 15 người. Thiếu: {missing_subjects}; ngoài danh sách: {extra_subjects}')
inventory = pd.DataFrame([{'sensor':group,'n_features':len(cols)} for group,cols in sensor_feature_groups.items()])
display(inventory)
save_table(inventory, 'sensor_feature_inventory.csv')
print(f'{len(df):,} cửa sổ | {len(subject_order)} người | {len(feature_cols)} feature (ngực {len(feature_groups["chest"])}, cổ tay {len(feature_groups["wrist"])})')
display(df.head())


## 3. Phân bố lớp và số cửa sổ theo người


In [ ]:
label_counts = df.label_original.value_counts().reindex([1,2,3], fill_value=0)
label_table = pd.DataFrame({'label_original':label_counts.index,
                            'class_name':[LABEL_NAMES[i] for i in label_counts.index],
                            'n_windows':label_counts.values})
label_table['percent'] = 100 * label_table.n_windows / len(df)
subject_label = pd.crosstab(df.subject_id, df.label_original).reindex(index=subject_order, columns=[1,2,3], fill_value=0)
subject_table = subject_label.rename(columns=LABEL_NAMES).copy()
subject_table['total'] = subject_table.sum(axis=1)
subject_table.index.name = 'subject_id'
display(label_table)
display(subject_table)
save_table(label_table, 'label_distribution.csv')
save_table(subject_table, 'windows_by_subject_and_label.csv', index=True)
if (subject_label == 0).any().any():
    warnings.warn('Có subject thiếu ít nhất một lớp; cần ghi rõ khi đánh giá LOSO.')

fig, axes = plt.subplots(1,2,figsize=(14,4.5))
axes[0].bar(label_table.class_name, label_table.n_windows, color=COLORS)
axes[0].set(title='Phân bố lớp — bước 5 giây', ylabel='Số cửa sổ')
for i, row in label_table.iterrows():
    axes[0].text(i, row.n_windows, f'{row.n_windows} ({row.percent:.1f}%)', ha='center', va='bottom')
subject_label.rename(columns=LABEL_NAMES).plot.bar(stacked=True, ax=axes[1], color=COLORS)
axes[1].set(title='Cửa sổ theo người và lớp', xlabel='Subject', ylabel='Số cửa sổ')
save_figure(fig, '01_label_subject_distribution.png')
fig, ax = plt.subplots(figsize=(11,4))
subject_label.div(subject_label.sum(axis=1), axis=0).rename(columns=LABEL_NAMES).plot.bar(stacked=True, ax=ax, color=COLORS)
ax.set(title='Tỷ lệ lớp trong từng người', xlabel='Subject', ylabel='Tỷ lệ', ylim=(0,1))
save_figure(fig, '02_class_proportions_by_subject.png')


## 4. Chất lượng dữ liệu
Kiểm tra NaN, Inf, khóa trùng, cột hằng và cờ chất lượng. Cờ
`finite_only_not_physiological_QA` chỉ cho biết kiểm tra tính hữu hạn,
không xác nhận tín hiệu đạt chất lượng sinh lý. Không tự điền thiếu hoặc xóa mẫu.


In [ ]:
values = df[feature_cols].to_numpy(dtype=float)
quality = pd.DataFrame({'feature':feature_cols,
    'nan_count':np.isnan(values).sum(axis=0),
    'inf_count':np.isinf(values).sum(axis=0),
    'n_unique_finite':[df[c].replace([np.inf,-np.inf],np.nan).nunique() for c in feature_cols]})
quality['nan_percent'] = 100 * quality.nan_count / len(df)
quality['constant_or_all_missing'] = quality.n_unique_finite <= 1
duplicates = df[df.duplicated(['subject_id','window_id'], keep=False)].copy()
quality_subject = df[['subject_id']].copy()
quality_subject['n_windows'] = 1
quality_subject['rows_with_nan'] = np.isnan(values).any(axis=1).astype(int)
quality_subject['rows_with_inf'] = np.isinf(values).any(axis=1).astype(int)
quality_subject['duplicate_key_rows'] = df.duplicated(['subject_id','window_id'],keep=False).astype(int)
quality_subject = quality_subject.groupby('subject_id').sum().reindex(subject_order)
flags = df.groupby(['subject_id','quality_flag'],dropna=False).size().rename('n_windows').reset_index()
display(quality)
display(quality_subject)
display(flags)
print('Số dòng thuộc nhóm khóa trùng:', len(duplicates))
save_table(quality, 'feature_quality.csv')
save_table(quality_subject, 'quality_by_subject.csv', index=True)
save_table(flags, 'quality_flags.csv')
save_table(duplicates, 'duplicate_keys.csv')
if len(duplicates) or np.isinf(values).any():
    warnings.warn('Có khóa trùng hoặc Inf: cần điều tra trước khi huấn luyện.')


## 5. Phân bố và điểm cực trị — toàn bộ feature

Histogram và boxplot chia trang, tối đa 16 feature/trang. Tất cả 112 cột
đều được vẽ; không giới hạn 21 ô như bản cũ. Danh sách hình và cột nằm
trong figure_manifest.csv để kiểm tra đủ phạm vi. Chỉ bỏ NaN/Inf khỏi hình,
không sửa bảng gốc. IQR chỉ đánh dấu để xem xét, không tự xóa ngoại lệ.


In [ ]:
finite = df[feature_cols].replace([np.inf,-np.inf],np.nan)
description = finite.describe().T
save_table(description, 'feature_descriptive_statistics.csv', index=True)
display(description)
by_label = finite.assign(label_original=df.label_original).groupby('label_original').agg(['count','mean','std','median'])
by_label.columns = ['__'.join(col) for col in by_label.columns]
save_table(by_label, 'feature_statistics_by_label.csv', index=True)

import inspect
PER_PAGE = 16
figure_manifest = []
for begin in range(0,len(feature_cols),PER_PAGE):
    page_cols = feature_cols[begin:begin+PER_PAGE]
    page = begin//PER_PAGE + 1
    for kind in ['histograms','boxplots']:
        nrows = (len(page_cols)+3)//4
        fig, axes = plt.subplots(nrows,4,figsize=(16,3*nrows),squeeze=False)
        for ax,col in zip(axes.flat,page_cols):
            groups = [finite.loc[df.label_original==k,col].dropna().to_numpy() for k in [1,2,3]]
            if kind == 'histograms':
                pooled = finite[col].dropna().to_numpy()
                edges = np.histogram_bin_edges(pooled,bins=20) if len(pooled) else np.linspace(0,1,21)
                for x,label,color in zip(groups,[1,2,3],COLORS):
                    if len(x): ax.hist(x,bins=edges,alpha=.45,color=color,label=LABEL_NAMES[label])
                ax.set_ylabel('Số cửa sổ')
            else:
                # Tương thích cả matplotlib trước và sau khi đổi labels → tick_labels.
                label_arg = 'tick_labels' if 'tick_labels' in inspect.signature(ax.boxplot).parameters else 'labels'
                ax.boxplot(groups,**{label_arg:list(LABEL_NAMES.values())},showfliers=True)
                ax.tick_params(axis='x',labelsize=7)
            ax.set_title(col,fontsize=8)
        for ax in axes.flat[len(page_cols):]: ax.set_visible(False)
        if kind=='histograms' and axes.flat[0].get_legend_handles_labels()[0]:
            axes.flat[0].legend(fontsize=7)
        fig.suptitle(f'{kind} — 60/5 — trang {page}',fontsize=12)
        name = f'{"03" if kind=="histograms" else "04"}_{kind}_page_{page:02d}.png'
        save_figure(fig,name)
        for col in page_cols:
            figure_manifest.append({'kind':kind,'page':page,'feature':col,'filename':name})
save_table(pd.DataFrame(figure_manifest), 'figure_manifest.csv')

q1,q3 = finite.quantile(.25),finite.quantile(.75)
iqr = q3-q1
outlier_mask = finite.lt(q1-1.5*iqr) | finite.gt(q3+1.5*iqr)
outlier_table = pd.DataFrame({'feature':feature_cols,'iqr':iqr.values,
    'flagged_count':outlier_mask.sum().values,'finite_count':finite.notna().sum().values})
outlier_table['flagged_percent'] = 100*outlier_table.flagged_count/outlier_table.finite_count.replace(0,np.nan)
display(outlier_table.sort_values('flagged_count',ascending=False))
save_table(outlier_table,'iqr_review_flags.csv')


## 6. Tương quan và khác biệt giữa người
Pearson mô tả liên hệ tuyến tính giữa đặc trưng, không chứng minh quan hệ
nhân quả. Không chọn/xóa cột dựa vào toàn bảng rồi báo kết quả test như một
đánh giá hoàn toàn độc lập; nếu chọn đặc trưng cho mô hình, thực hiện trong fold train.


In [ ]:
corr = finite.corr(method='pearson')
save_table(corr,'pearson_correlation.csv',index=True)
for group in ['combined','chest','wrist']:
    cols = feature_groups[group]
    sub = corr.loc[cols,cols]
    size = max(10,min(24,len(cols)*.20))
    fig,ax = plt.subplots(figsize=(size,size*.88))
    im = ax.imshow(sub,cmap='coolwarm',vmin=-1,vmax=1)
    ax.set_xticks(range(len(cols)),cols,rotation=90,fontsize=5)
    ax.set_yticks(range(len(cols)),cols,fontsize=5)
    ax.set_title(f'Tương quan Pearson — {group} — raw 60/5')
    fig.colorbar(im,ax=ax,label='r')
    save_figure(fig,f'05_pearson_heatmap_{group}.png')
pairs = [{'feature_a':a,'feature_b':b,'pearson_r':corr.loc[a,b]}
    for i,a in enumerate(feature_cols) for b in feature_cols[i+1:]
    if pd.notna(corr.loc[a,b]) and abs(corr.loc[a,b])>=.95]
high_corr = pd.DataFrame(pairs,columns=['feature_a','feature_b','pearson_r'])
display(high_corr.head(30))
save_table(high_corr,'high_correlation_pairs.csv')
subject_means = finite.assign(subject_id=df.subject_id).groupby('subject_id').mean().reindex(subject_order)
save_table(subject_means,'mean_features_by_subject.csv',index=True)
# Một feature minh họa mỗi cảm biến, gồm đủ 6 ngực và 4 cổ tay.
representatives = ['chest_ecg_std','chest_eda_mean','chest_emg_rms',
    'chest_resp_std','chest_temp_mean','chest_acc_magnitude_mean',
    'wrist_bvp_std','wrist_eda_mean','wrist_temp_mean','wrist_acc_magnitude_mean']
fig,axes = plt.subplots(5,2,figsize=(16,17))
for ax,col in zip(axes.flat,representatives):
    ax.bar(subject_order,subject_means[col],color='#4c78a8')
    ax.set(title=col,xlabel='Subject',ylabel='Trung bình đặc trưng')
    ax.tick_params(axis='x',rotation=90,labelsize=8)
save_figure(fig,'06_subject_differences.png')


## 7. Khảo sát tín hiệu thô — ngực và cổ tay

Giữ nguyên thiết lập đường dẫn PKL và công tắc RUN_RAW_EDA.
Kiểm tra đủ 10 kênh theo tần số gốc; cổ tay không so số mẫu trực tiếp với
nhãn 700 Hz. Báo số mẫu kỳ vọng theo thời lượng nhãn, sai lệch mẫu và thời
lượng. Chênh lệch không quá một mẫu của kênh được coi là khớp trong kiểm tra
sơ bộ này, không phải chứng nhận đồng bộ hoặc QA sinh lý.
Vẽ một đoạn đồng nhất 10 giây mỗi trạng thái cho từng kênh của S3.
Chỉ mở PKL WESAD tin cậy; có thể tắt RUN_RAW_EDA nếu chỉ phân tích feature.


In [ ]:
RUN_RAW_EDA = True
RAW_ROOT_OVERRIDE = None  # Ví dụ: r'D:/Datasets/WESAD/WESAD'
RAW_ROOT = PROJECT_ROOT / 'data/raw/WESAD/WESAD'
if RAW_ROOT_OVERRIDE:
    RAW_ROOT = Path(RAW_ROOT_OVERRIDE).expanduser().resolve()
elif RUN_RAW_EDA and (PROJECT_ROOT / 'configs/local.yaml').is_file():
    import yaml
    local_config = yaml.safe_load((PROJECT_ROOT / 'configs/local.yaml').read_text(encoding='utf-8'))
    if not isinstance(local_config, dict) or not local_config.get('data_root'):
        raise ValueError('configs/local.yaml cần có data_root.')
    raw_path = Path(local_config['data_root']).expanduser()
    RAW_ROOT = raw_path if raw_path.is_absolute() else (PROJECT_ROOT / raw_path).resolve()
raw_label_table = None
raw_quality_table = None
RAW_SUBJECTS = EXPECTED_SUBJECTS
ILLUSTRATION_SUBJECT = 'S3'

if RUN_RAW_EDA:
    import pickle
    import gc
    raw_labels,raw_quality = [],[]
    for subject in RAW_SUBJECTS:
        path = RAW_ROOT / subject / f'{subject}.pkl'
        if not path.is_file():
            warnings.warn(f'Bỏ qua file không có: {path}')
            continue
        with path.open('rb') as f:
            raw = pickle.load(f,encoding='latin1')
        labels = np.asarray(raw['label'])
        if labels.ndim!=1 or not labels.size or not np.isfinite(labels).all():
            raise ValueError(f'{subject}: nhãn phải hữu hạn, một chiều và không rỗng.')
        unique,counts = np.unique(labels,return_counts=True)
        raw_labels.extend({'subject_id':subject,'label':int(k),'n_samples':int(v),
            'seconds_at_700hz':int(v)/700} for k,v in zip(unique,counts))
        duration = len(labels)/700
        for device,sensors in SENSOR_SPECS.items():
            for sensor,(rate,axes) in sensors.items():
                values_raw = raw.get('signal',{}).get(device,{}).get(sensor)
                expected_n = (len(labels)*rate+699)//700
                base = {'subject_id':subject,'device':device,'sensor':sensor,
                        'sampling_rate_hz':rate,'expected_samples':expected_n}
                if values_raw is None:
                    raw_quality.append({**base,'present':False,'shape':'missing','n_samples':0,
                        'shape_ok':False,'nan_values':0,'inf_values':0,'duration_seconds':0.,
                        'sample_difference':-expected_n,'duration_matches_labels':False})
                    warnings.warn(f'{subject}: thiếu {device}/{sensor}')
                    continue
                x = np.asarray(values_raw)
                shape_ok = ((x.ndim==1 and axes==1) or (x.ndim==2 and x.shape[1]==axes))
                if x.ndim==0 or not np.issubdtype(x.dtype,np.number):
                    raise ValueError(f'{subject} {device}/{sensor}: tín hiệu phải là mảng số.')
                n_samples = len(x)
                raw_quality.append({**base,'present':True,'shape':str(x.shape),'n_samples':n_samples,
                    'shape_ok':bool(shape_ok),'nan_values':int(np.isnan(x).sum()),
                    'inf_values':int(np.isinf(x).sum()),'duration_seconds':n_samples/rate,
                    'sample_difference':n_samples-expected_n,
                    'duration_matches_labels':bool(abs(n_samples/rate-duration)<=1/rate+1e-9)})
        if subject==ILLUSTRATION_SUBJECT:
            example_starts = {}
            for label in [1,2,3]:
                mask = labels==label
                edges = np.diff(np.r_[False,mask,False].astype(int))
                starts,ends = np.flatnonzero(edges==1),np.flatnonzero(edges==-1)
                starts = starts[(ends-starts)>=10*700]
                example_starts[label] = int(starts[0]) if len(starts) else None
            for device,sensors in SENSOR_SPECS.items():
                fig,axes = plt.subplots(len(sensors),3,figsize=(15,2.7*len(sensors)),squeeze=False)
                for i,(sensor,(rate,n_axes)) in enumerate(sensors.items()):
                    values_raw = raw.get('signal',{}).get(device,{}).get(sensor)
                    for j,label in enumerate([1,2,3]):
                        ax = axes[i,j]
                        ax.set_title(f'{subject} | {device}/{sensor} | {LABEL_NAMES[label]}',fontsize=9)
                        ax.set_xlabel('Giây trong đoạn')
                        start = example_starts[label]
                        if values_raw is None or start is None:
                            ax.text(.5,.5,'Thiếu kênh hoặc đoạn đủ 10 giây',ha='center',transform=ax.transAxes)
                            continue
                        first = (start*rate+699)//700
                        last = ((start+10*700)*rate+699)//700
                        if last>len(values_raw):
                            ax.text(.5,.5,'Kênh không phủ đủ đoạn',ha='center',transform=ax.transAxes)
                            continue
                        segment = np.asarray(values_raw)[first:last]
                        if segment.ndim>2:
                            continue
                        ax.plot(np.arange(len(segment))/rate,segment,linewidth=.6)
                        if sensor=='ACC' and segment.ndim==2 and segment.shape[1]==3:
                            ax.legend(['x','y','z'],fontsize=7)
                save_figure(fig,f'07_raw_signals_{subject}_{device}.png')
        # Không giữ view của bản ghi cũ khi đọc PKL người tiếp theo.
        del raw,labels
        for variable in ['x','values_raw','segment','mask','edges','starts','ends']:
            globals().pop(variable,None)
        gc.collect()
    if not raw_labels:
        raise FileNotFoundError('Không đọc được PKL nào; kiểm tra đường dẫn raw hiện có hoặc tắt RUN_RAW_EDA khi chỉ cần bảng feature.')
    raw_label_table = pd.DataFrame(raw_labels)
    raw_quality_table = pd.DataFrame(raw_quality)
    display(raw_label_table)
    display(raw_quality_table)
    save_table(raw_label_table,'raw_label_distribution.csv')
    save_table(raw_quality_table,'raw_signal_quality.csv')
else:
    print('Không chạy khảo sát raw trong lần này; phần EDA bảng feature vẫn thực hiện đầy đủ.')


## 8. Tóm tắt tự động và lưu thông tin lần chạy


In [ ]:
nonzero = label_counts[label_counts>0]
ratio = float(nonzero.max()/nonzero.min()) if len(nonzero)==3 else None
summary = {
    'created_at_utc':datetime.now(timezone.utc).isoformat(),
    'input_file':str(FEATURE_PATH),
    'input_sha256':hashlib.sha256(FEATURE_PATH.read_bytes()).hexdigest(),
    'scope':'all chest and wrist raw statistics; window 60s; step 5s; majority >=0.90',
    'processing_metadata_verified':processing is not None,
    'processing':processing,
    'n_windows':len(df),'n_subjects':len(subject_order),'n_features':len(feature_cols),
    'subjects':subject_order, 'missing_subjects':missing_subjects, 'extra_subjects':extra_subjects,
    'feature_groups':{group:len(cols) for group,cols in feature_groups.items()},
    'schema_present':schema is not None,
    'label_counts':{str(k):int(v) for k,v in label_counts.items()},
    'largest_smallest_class_ratio':ratio,
    'nan_feature_values':int(np.isnan(values).sum()),
    'inf_feature_values':int(np.isinf(values).sum()),
    'duplicate_key_rows':len(duplicates),
    'constant_or_all_missing_features':quality.loc[quality.constant_or_all_missing,'feature'].tolist(),
    'window_seconds':WINDOW_SECONDS, 'step_seconds':STEP_SECONDS,
    'label_rule':'majority', 'min_label_fraction':MIN_LABEL_FRACTION,
    'observed_min_label_fraction':float(df.label_fraction.min()),
    'observed_max_label_fraction':float(df.label_fraction.max()),
    'raw_eda_enabled':RUN_RAW_EDA,
    'raw_subjects_checked':sorted(raw_label_table.subject_id.unique().tolist()) if raw_label_table is not None else [],
    'interpretation_limit':'Exploratory full-table analysis; no model performance or physiological QA claim.'
}
(TABLE_DIR / 'eda_summary.json').write_text(json.dumps(summary,ensure_ascii=False,indent=2),encoding='utf-8')
print(f'Có {len(df)} cửa sổ từ {len(subject_order)} người, với {len(feature_cols)} đặc trưng.')
print('Số mẫu Baseline/Stress/Amusement:', label_counts.tolist())
print('Tỷ lệ lớp lớn nhất / nhỏ nhất:', round(ratio,2) if ratio is not None else 'Thiếu ít nhất một lớp')
print('NaN:', summary['nan_feature_values'], '| Inf:', summary['inf_feature_values'], '| Dòng khóa trùng:',len(duplicates))


print('Bảng:', TABLE_DIR)
print('Hình:', FIG_DIR)


## 9. Nhận xét theo kết quả bộ 60/5

Ô dưới tạo nhận xét từ dữ liệu vừa chạy và lưu `eda_observations.md`.
Không dùng số liệu 513 cửa sổ hay tỷ lệ lớp của bộ 60/60 cũ.
Sau khi xem hình, bổ sung diễn giải hình dạng phân bố/tín hiệu; nhận xét
tự động không thay thế kiểm tra chất lượng sinh lý hoặc kết quả mô hình.


In [ ]:
from IPython.display import Markdown
lines = [
    '## Nhận xét EDA — cửa sổ 60 giây, bước 5 giây',
    '',
    f'Bảng hiện có **{len(df):,} cửa sổ**, **{len(subject_order)} người** và **{len(feature_cols)} đặc trưng** thống kê raw ngực và cổ tay.',
    'Cấu hình mục tiêu là majority ≥90%. Có thể có tối đa 10% mẫu khác nhãn trong cửa sổ hợp lệ.',
    ('Đã đối chiếu cấu hình trích xuất từ config.json; đây là kiểm tra metadata, chưa tính lại feature/nhãn từ PKL.' if processing is not None else
     '**Hạn chế:** thiếu config.json; lưới thời gian và tỷ lệ nhãn trong CSV chưa đủ xác minh cấu hình tạo dữ liệu.'),
    '', '### Phân bố lớp',
]
for row in label_table.itertuples(index=False):
    lines.append(f'- {row.class_name}: {row.n_windows:,} cửa sổ ({row.percent:.2f}%).')
lines.append(f'Số cửa sổ theo người dao động {int(subject_table.total.min())}–{int(subject_table.total.max())}.')
if ratio is not None:
    lines.append(f'Tỷ lệ số cửa sổ lớp lớn nhất/nhỏ nhất là {ratio:.2f}.')
missing_class_subjects = subject_label.index[(subject_label == 0).any(axis=1)].tolist()
lines.append('Các subject thiếu ít nhất một lớp: ' + (', '.join(missing_class_subjects) if missing_class_subjects else 'không có trong bảng hiện tại') + '.')
if missing_subjects or extra_subjects:
    lines.append(f'**Phạm vi chưa khớp 15 người:** thiếu {missing_subjects}; ngoài danh sách {extra_subjects}.')
lines.extend(['', '### Chất lượng và điểm cần kiểm tra',
    f'Có {summary["nan_feature_values"]} giá trị NaN, {summary["inf_feature_values"]} giá trị Inf và {len(duplicates)} dòng thuộc nhóm khóa trùng.',
    f'Cột hằng/thiếu toàn bộ: {summary["constant_or_all_missing_features"]}.',
    f'label_fraction quan sát từ {df.label_fraction.min():.4f} đến {df.label_fraction.max():.4f}.',
    'quality_flag phản ánh phạm vi kiểm tra của extractor; tính hữu hạn không chứng minh chất lượng sinh lý.'])
for row in outlier_table.sort_values('flagged_count',ascending=False).head(3).itertuples(index=False):
    lines.append(f'- {row.feature}: {row.flagged_count} giá trị bị đánh dấu IQR ({row.flagged_percent:.2f}% trên giá trị hữu hạn).')
lines.append('Đánh dấu IQR phục vụ xem xét; chưa tự loại cửa sổ hoặc kết luận đó là lỗi.')
lines.extend(['', '### Tương quan'])
if high_corr.empty:
    lines.append('Không có cặp đạt |r| ≥0.95 trong bảng hiện tại.')
else:
    for row in high_corr.reindex(high_corr.pearson_r.abs().sort_values(ascending=False).index).head(5).itertuples(index=False):
        lines.append(f'- {row.feature_a} và {row.feature_b}: r = {row.pearson_r:.6f}.')
lines.append('Nếu chọn đặc trưng cho mô hình, học lựa chọn trong train của từng fold; không chọn trên toàn bảng rồi báo điểm test như đánh giá độc lập.')
lines.extend(['', '### Tín hiệu thô'])
if raw_quality_table is not None:
    checked = sorted(raw_quality_table.subject_id.unique().tolist())
    lines.append(f'Đã kiểm tra {len(checked)} người và {len(raw_quality_table)} cặp người–thiết bị–tín hiệu ngực/cổ tay trong lần chạy này.')
    lines.append(f'Tổng NaN: {int(raw_quality_table.nan_values.sum())}; tổng Inf: {int(raw_quality_table.inf_values.sum())}; số cặp không khớp thời lượng nhãn trong dung sai một mẫu: {int((~raw_quality_table.duration_matches_labels).sum())}.')
    lines.append(f'Số kênh bị thiếu: {int((~raw_quality_table.present).sum())}; số cặp sai shape: {int((~raw_quality_table.shape_ok).sum())}.')
    missing_raw = sorted(set(RAW_SUBJECTS) - set(checked))
    if missing_raw:
        lines.append(f'Chưa khảo sát raw của: {missing_raw}.')
else:
    lines.append('Không chạy khảo sát raw trong lần chạy này; không khẳng định đã kiểm tra đủ tín hiệu thô.')
lines.extend(['', '### Hướng mô hình hóa',
    f'Chỉ dùng {len(feature_cols)} feature trong schema làm X (ngực {len(feature_groups["chest"])}, cổ tay {len(feature_groups["wrist"])}); giữ subject_id cho LOSO, không đưa metadata hoặc nhãn vào X.',
    'Stress là lớp 1 trong bài toán nhị phân; Baseline và Amusement là Non-stress (0).',
    'Imputer/scaler/selection/resampling học trong train của từng fold. Tuning chỉ dùng subject thuộc outer train.',
    'Cửa sổ liên tiếp chồng lấn 55 giây, không coi chúng là quan sát độc lập. Không chia ngẫu nhiên các dòng train/test.',
    'EDA không chứng minh hiệu năng hay khả năng khái quát của KNN/SVM. Báo cáo giữa kỳ tiếp tục bổ sung baseline và mô hình mặc định; tuning và kiểm định theo kế hoạch nhóm.'])
observations = '\n\n'.join(lines)
(TABLE_DIR / 'eda_observations.md').write_text(observations, encoding='utf-8')
display(Markdown(observations))


## 10. Bàn giao

Giữ nguyên thư mục output đã thiết lập ở mục 1. Sau khi chạy dữ liệu thật,
lưu notebook có output, các bảng/hình, eda_summary.json, eda_observations.md
và figure_manifest.csv. Tất cả feature có histogram/boxplot theo trang;
heatmap và bảng tương quan bao phủ combined/ngực/cổ tay.

Các tên hình mới theo trang và nhóm; hình cũ trong cùng thư mục không tự
bị xóa. Khi nộp, chọn hình từ lần chạy mới, đối chiếu figure_manifest.csv;
không dùng hình/nhận xét 21 feature cũ. Không tự khẳng định hoàn tất G1–G3
hoặc QA sinh lý chỉ vì notebook chạy xong.
